# Доразметка RAG-датасета по готовому ответу

Человек заполняет шесть столбцов: `question_id`, `question`, `multiproject_flag`, `answerable`, `relevant_answer`, `key_facts`.
Ноутбук их не переписывает и генератор не вызывает.

`multiproject_flag` — режим поиска из веб-интерфейса: `one` — один проект, `several` — выбранные, `all` — все неархивные.
Если `answerable` равен `false`, списки id и источников остаются пустыми: генератор должен ответить, что в контексте нет информации.
Если `true`, ответ режется на ключевые факты, а если их нет — на предложения. Ретривер ищет кандидатов по каждому куску. Близость карточки — лучшее совпадение с одним куском, той же моделью эмбеддингов, что в основном поиске. В файл попадают карточки с близостью не ниже 0.8. Карточка с одним фактом не штрафуется за то, что остальные факты лежат в других карточках.

Для `one` и `several` нужен столбец `projects`: какие проекты были выбраны в интерфейсе. В итоговый файл он не пишется, он только задаёт область поиска.

Ячейки запускайте сверху вниз. Тот же `question_id` при повторной записи заменяет старую строку. `reviewed` остаётся `false`, пока id не проверены вручную.


## 1. Проекты

`graph_id` из файла проектов — это `user_id` карточек в Neo4j.


In [ ]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "storage" / "web_projects.json").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PROJECTS_FILE = ROOT / "storage" / "web_projects.json"


def load_projects():
    raw = json.loads(PROJECTS_FILE.read_text(encoding="utf-8"))
    items = raw["projects"] if isinstance(raw, dict) else raw
    return [item for item in items if item.get("graph_id")]


projects = load_projects()
projects_view = pd.DataFrame([
    {
        "slug": item.get("slug"),
        "name": item.get("name"),
        "graph_id": item.get("graph_id"),
        "archived": bool(item.get("archived")),
    }
    for item in projects
])
print(f"проектов: {len(projects)}, без архива: {sum(not item.get('archived') for item in projects)}")
display(projects_view)
assert projects


## 2. Таблица вопрос–ответ

Положите csv или xlsx. Столбцы, которые заполняет человек:

- `question_id` — id вопроса
- `question` — вопрос
- `multiproject_flag` — `one`, `several` или `all`
- `answerable` — `true`, если ответ есть в графе, иначе `false`
- `relevant_answer` — правильный ответ генератора
- `key_facts` — факты для полноты, через точку с запятой, с новой строки или JSON-списком

Для `one` и `several` добавьте `projects`: slug, `graph_id` или название через запятую. Это не эталон, а область, которую выбрали в интерфейсе. Для `all` столбец не нужен.

Если файла нет, берётся пример из этой ячейки.


In [ ]:
QA_PATH = ROOT / "eval" / "synthetic_datasets" / "graphrag" / "qa_input.xlsx"
FLAGS = {"one", "several", "all"}
EMPTY_ANSWER = "В контексте нет информации по этому вопросу."

EXAMPLE = [
    {
        "question_id": "q001",
        "question": "Какой статус у сервиса оплаты?",
        "multiproject_flag": "one",
        "answerable": "true",
        "relevant_answer": "Сервис оплаты в эксплуатации.",
        "key_facts": "Сервис оплаты в эксплуатации.",
        "projects": "b2b_аудит",
    },
]


def cell(value):
    if value is None:
        return ""
    text = str(value).strip()
    return "" if text.lower() == "nan" else text


def parse_bool(value):
    text = cell(value).lower()
    if text in {"true", "1", "yes", "да", "y"}:
        return True
    if text in {"false", "0", "no", "нет", "n"}:
        return False
    raise SystemExit(f"answerable должен быть true или false, получено: {value}")


def parse_facts(value):
    if isinstance(value, list):
        return [cell(item) for item in value if cell(item)]
    text = cell(value)
    if not text:
        return []
    if text.startswith("["):
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            parsed = None
        if isinstance(parsed, list):
            return [cell(item) for item in parsed if cell(item)]
    return [part.strip() for part in re.split(r"\n|;", text) if part.strip()]


def load_qa(path):
    path = Path(path)
    frame = pd.read_excel(path) if path.suffix.lower() in {".xlsx", ".xls"} else pd.read_csv(path)
    frame.columns = [str(col).strip() for col in frame.columns]
    if "multiproject_flag" not in frame.columns and "flag" in frame.columns:
        frame = frame.rename(columns={"flag": "multiproject_flag"})
    if "relevant_answer" not in frame.columns and "answer" in frame.columns:
        frame = frame.rename(columns={"answer": "relevant_answer"})
    missing = [
        col for col in ("question_id", "question", "multiproject_flag", "answerable", "relevant_answer", "key_facts")
        if col not in frame.columns
    ]
    if missing:
        raise SystemExit(f"В таблице нет столбцов: {', '.join(missing)}")

    rows = []
    for record in frame.to_dict(orient="records"):
        question = cell(record.get("question"))
        if not question:
            continue
        flag = cell(record.get("multiproject_flag")).lower()
        if flag not in FLAGS:
            raise SystemExit(f"У вопроса «{question}» multiproject_flag должен быть one, several или all.")
        answerable = parse_bool(record.get("answerable"))
        answer = cell(record.get("relevant_answer"))
        if not answerable and not answer:
            answer = EMPTY_ANSWER
        projects_text = record.get("projects")
        if not cell(projects_text):
            projects_text = record.get("search_project_ids") or ""
        rows.append({
            "question_id": cell(record.get("question_id")),
            "question": question,
            "flag": flag,
            "answerable": answerable,
            "answer": answer,
            "facts": [] if not answerable else parse_facts(record.get("key_facts")),
            "projects": cell(projects_text),
        })
    if not rows:
        raise SystemExit("В таблице нет вопросов.")
    return rows


assert parse_bool("true") and not parse_bool("false")
assert parse_facts('["статус сервиса", "в эксплуатации"]') == ["статус сервиса", "в эксплуатации"]
assert parse_facts("статус сервиса; в эксплуатации") == ["статус сервиса", "в эксплуатации"]
print("разбор answerable и key_facts прошёл")

if QA_PATH.exists():
    QUESTIONS = load_qa(QA_PATH)
    print("таблица:", QA_PATH)
else:
    QUESTIONS = []
    for raw in EXAMPLE:
        answerable = parse_bool(raw["answerable"])
        answer = cell(raw["relevant_answer"])
        if not answerable and not answer:
            answer = EMPTY_ANSWER
        QUESTIONS.append({
            "question_id": raw["question_id"],
            "question": raw["question"],
            "flag": raw["multiproject_flag"],
            "answerable": answerable,
            "answer": answer,
            "facts": [] if not answerable else parse_facts(raw["key_facts"]),
            "projects": raw.get("projects") or "",
        })
    print("Файла нет, взята таблица из ячейки.")
    print("Чтобы подставить свою, сохраните её сюда:", QA_PATH)

for number, item in enumerate(QUESTIONS, start=1):
    if not item["question_id"]:
        item["question_id"] = f"q{number:03d}"

view = pd.DataFrame([
    {
        "question_id": item["question_id"],
        "multiproject_flag": item["flag"],
        "answerable": item["answerable"],
        "question": item["question"],
        "relevant_answer": item["answer"],
        "key_facts": item["facts"],
        "projects": item["projects"],
    }
    for item in QUESTIONS
])
display(view)
assert len({item["question_id"] for item in QUESTIONS}) == len(QUESTIONS), "question_id повторяется"


## 3. Где искать

`one` ищет внутри одного проекта, как страница проекта.
`several` ищет по перечисленным проектам, как совместный поиск.
`all` ищет по всем неархивным, как общий граф. Список проектов при `all` область не сужает.


In [ ]:
def resolve_projects(flag, text, projects):
    names = [part.strip() for part in re.split(r"[,;]", text or "") if part.strip()]
    if flag == "one" and len(names) != 1:
        raise SystemExit("Для one укажите один проект в столбце projects.")
    if flag == "several" and len(names) < 2:
        raise SystemExit("Для several укажите хотя бы два проекта в столбце projects.")
    found = []
    for name in names:
        key = name.casefold()
        match = next(
            (
                item for item in projects
                if key in {
                    str(item.get("slug") or "").casefold(),
                    str(item.get("graph_id") or "").casefold(),
                    str(item.get("name") or "").casefold(),
                }
            ),
            None,
        )
        if match is None:
            known = ", ".join(item.get("slug") or item["graph_id"] for item in projects)
            raise SystemExit(f"Проект не найден: {name}. Известные slug: {known}")
        found.append(match)
    return found


labels = {
    item["graph_id"]: item.get("name") or item.get("slug") or item["graph_id"]
    for item in projects
}
active = [item for item in projects if not item.get("archived")]

for item in QUESTIONS:
    # Тот же выбор области, что в веб-интерфейсе: страница проекта, совместный поиск или общий граф.
    if item["flag"] == "all":
        item["chosen"] = active
        where = "все неархивные проекты"
    else:
        if not (item.get("projects") or "").strip():
            raise SystemExit(
                f"{item['question_id']}: для режима {item['flag']} укажите проекты, как в веб-интерфейсе."
            )
        item["chosen"] = resolve_projects(item["flag"], item["projects"], projects)
        where = "один проект" if item["flag"] == "one" else "выбранные проекты"
    item["graph_ids"] = [project["graph_id"] for project in item["chosen"]]
    print(item["question_id"], item["flag"], "->", where, f"({len(item['chosen'])})")
    for project in item["chosen"]:
        print(f"  {project.get('name')} | {project['graph_id']}")
    if item["flag"] == "one":
        assert len(item["graph_ids"]) == 1
    elif item["flag"] == "several":
        assert len(item["graph_ids"]) >= 2
assert active, "нет неархивных проектов"


## 4. Тип источника

Папка из списка наблюдения помечает файл внутри неё как `directory`. Одиночный путь — `file`. Ссылка — `confluence`. Пустой источник — `text`.


In [ ]:
from storage.postgres.db_connect import list_watch_sources


def folder_roots():
    try:
        rows = list_watch_sources(watch_only=False)
    except Exception as exc:
        print(f"Папки не прочитаны, пути будут помечены как file: {exc}")
        return []
    return [
        Path(row["source_path"].strip())
        for row in rows
        if row.get("source_kind") == "folder" and (row.get("source_path") or "").strip()
    ]


folders = folder_roots()
print(f"папок: {len(folders)}")
for path in folders:
    print(" -", path)


In [ ]:
def source_type(source, folders):
    value = (source or "").strip() or "text"
    if value == "text":
        return "text"
    if value.lower().startswith(("http://", "https://")):
        return "confluence"
    path = Path(value)
    for root in folders:
        try:
            path.relative_to(root)
            return "directory"
        except ValueError:
            continue
    return "file"


samples = ["text", "https://conf.example/page", "/tmp/one.pdf"]
if folders:
    samples.append(str(folders[0] / "a.xlsx"))
for sample in samples:
    print(f"{source_type(sample, folders):12} {sample}")


## 5. Порог

Ответ сравнивается по кускам: ключевые факты, а если их нет — предложения. Карточка берётся, если хотя бы один кусок близок к её тексту не ниже 0.8. Совпадение отдельных слов не используется. Если такой карточки нет, список id пустой.


In [ ]:
def card_text(card):
    topic = (card.topic or "").strip()
    body = " ".join((card.content or "").split())
    return f"{topic}\n{body}".strip() if topic and body else (topic or body)


def keep_sure(scored, minimum):
    ranked = sorted(scored, key=lambda pair: pair[1], reverse=True)
    return [pair[0] for pair in ranked if pair[1] >= minimum]


def split_parts(text):
    text = " ".join((text or "").split())
    if not text:
        return []
    pieces = [part.strip() for part in re.split(r"(?<=[.!?\u2026])\s+|\n+", text) if part.strip()]
    return pieces or [text]


def answer_parts(item):
    # Кусок — один факт. Целый ответ из нескольких фактов не сравнивается одной строкой.
    parts = []
    for fact in item["facts"]:
        parts.extend(split_parts(fact))
    if not parts:
        parts.extend(split_parts(item["answer"]))
    full = " ".join((item["answer"] or "").split())
    if full and full not in parts:
        parts.append(full)
    result = []
    for part in parts:
        if part not in result:
            result.append(part)
    return result


class _Card:
    def __init__(self, zettel_id):
        self.zettel_id = zettel_id


high = _Card("да")
border = _Card("граница")
low = _Card("нет")
kept = keep_sure([(low, 0.79), (high, 0.91), (border, 0.8)], 0.8)
assert [card.zettel_id for card in kept] == ["да", "граница"]
assert keep_sure([(low, 0.79)], 0.8) == []
print("порог 0.8: 0.91 и 0.80 остаются, 0.79 нет")

one_part = answer_parts({"answer": "Сервис оплаты в эксплуатации.", "facts": ["Сервис оплаты в эксплуатации."]})
many_parts = answer_parts({
    "answer": "Сервис оплаты в эксплуатации. Владелец — команда биллинга.",
    "facts": ["Сервис оплаты в эксплуатации.", "Владелец — команда биллинга."],
})
assert one_part == ["Сервис оплаты в эксплуатации."]
assert many_parts[:2] == ["Сервис оплаты в эксплуатации.", "Владелец — команда биллинга."]
print("ответ из двух карточек сравнивается по кускам:", many_parts[:2])


## 6. Поиск кандидатов

Ищется тем же ретривером и в той же области, что в веб-интерфейсе. Запрос каждого куска ответа, не вопрос. Это только кандидаты. Порог 0.8 применяется в следующей ячейке.


In [ ]:
from config.settings import settings
from zettelkasten.graph_rag import GraphRetriever
from zettelkasten.linker import LocalEmbeddingModel

retriever = GraphRetriever(
    embedding_model=LocalEmbeddingModel(model_name=settings.embedding_model_name),
    search_limit=8,
)
print("порог похожести:", settings.graphrag_similarity_threshold)
print("карточек на запрос:", retriever.search_limit)

RERANK_MIN = 0.8
print("в эталон берём близость ответа и карточки не ниже", RERANK_MIN)


In [ ]:
def snippet(card, limit=180):
    text = card_text(card)
    if len(text) <= limit:
        return text
    return text[: limit - 1] + "…"


def retrieve(query, item):
    # one — внутри одного проекта. several и all — по списку graph_id.
    query = " ".join((query or "").split())
    if item["flag"] == "one":
        return retriever.retrieve(item["graph_ids"][0], query)
    return retriever.retrieve("__all__", query, user_ids=item["graph_ids"])


def add_found(pool, found):
    for place, cards in (("entry", found.entry_points), ("expanded", found.expanded_nodes)):
        for card in cards:
            saved = pool.get(card.zettel_id)
            score = float(card.similarity or 0)
            if saved is None:
                pool[card.zettel_id] = {"card": card, "place": place, "score": score}
            else:
                saved["score"] = max(saved["score"], score)
                if place == "entry":
                    saved["place"] = "entry"


hit_rows = []
for item in QUESTIONS:
    item["pool"] = {}
    if not item["answerable"]:
        item["parts"] = []
        print(item["question_id"], "answerable=false, поиск пропущен")
        continue
    parts = answer_parts(item)
    item["parts"] = parts
    print(f"{item['question_id']}: кусков ответа {len(parts)}")
    for part in parts:
        found = retrieve(part[:500], item)
        add_found(item["pool"], found)
        print(
            f"  «{part[:80]}»: точек входа {len(found.entry_points)}, "
            f"раскрыто {len(found.expanded_nodes)}"
        )
    ranked = sorted(item["pool"].values(), key=lambda info: -info["score"])
    print(f"{item['question_id']}: кандидатов {len(ranked)}")
    for info in ranked[:12]:
        card = info["card"]
        hit_rows.append({
            "question_id": item["question_id"],
            "place": info["place"],
            "retrieval": round(info["score"], 4),
            "luhmann_id": card.luhmann_id,
            "project": labels.get(card.user_id or "", card.user_id),
            "source": (card.source_input or "").strip() or "text",
            "snippet": snippet(card),
        })

display(pd.DataFrame(hit_rows))


## 7. Отбор по ответу

Каждая карточка сравнивается с каждым куском ответа той же моделью эмбеддингов, что и основной поиск. Близость карточки — лучший кусок. Остаются карточки с близостью не ниже 0.8.


In [ ]:
def cosine(left, right):
    dot = sum(a * b for a, b in zip(left, right))
    left_norm = sum(a * a for a in left) ** 0.5
    right_norm = sum(b * b for b in right) ** 0.5
    if left_norm < 1e-10 or right_norm < 1e-10:
        return 0.0
    return dot / (left_norm * right_norm)


def score_cards(parts, cards):
    # Тот же счёт, что у векторного поиска. Близость карточки — лучший кусок ответа.
    parts = [part for part in parts if part.strip()]
    if not cards or not parts:
        return []
    query_vectors = [retriever.embedding_model.embed_query(part) for part in parts]
    vectors = retriever.embedding_model.embed_passages_batch([card_text(card)[:1500] for card in cards])
    scored = []
    for card, vector in zip(cards, vectors):
        best_score = 0.0
        best_part = parts[0]
        for part, query_vector in zip(parts, query_vectors):
            value = cosine(query_vector, vector)
            if value > best_score:
                best_score = value
                best_part = part
        scored.append((card, round(best_score, 4), best_part))
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored


for item in QUESTIONS:
    if not item["answerable"]:
        item["gold"] = []
        item["scores"] = {}
        item["matched"] = {}
        print(item["question_id"], "answerable=false, список id пустой")
        continue
    cards = [info["card"] for info in item["pool"].values()]
    scored = score_cards(item["parts"], cards)
    item["scores"] = {card.zettel_id: score for card, score, part in scored}
    item["matched"] = {card.zettel_id: part for card, score, part in scored}
    gold = keep_sure(scored, RERANK_MIN)
    item["gold"] = gold
    print(f"{item['question_id']}: берём {len(gold)} из {len(scored)}")
    for card, score, part in scored[:12]:
        mark = "берём" if score >= RERANK_MIN else "нет"
        print(f"  {mark} {score:.3f} [{card.luhmann_id}] {card.topic} | {part[:80]}")
    if not gold:
        best = scored[0][1] if scored else 0
        print(f"  нет карточки с близостью от {RERANK_MIN}, лучшая {best}")


## 8. Строка датасета

В файле остаются ответ человека и то, по чему видно, тот ли узел найден: номер Luhmann, id, близость, текст узла и источник. Остальные карточки в файл не пишутся.


In [ ]:
def as_json(values):
    return json.dumps(list(values), ensure_ascii=False)


def uniq(values):
    result = []
    for value in values:
        if value and value not in result:
            result.append(value)
    return result


def node_text(card, limit=2500):
    text = card_text(card)
    if len(text) <= limit:
        return text
    return text[: limit - 1] + "…"


def relevant_texts(cards, scores, matched):
    blocks = []
    for card in cards:
        mark = card.luhmann_id or card.zettel_id
        piece = matched.get(card.zettel_id, "")
        head = f"[{mark}] {scores.get(card.zettel_id, '')}"
        if piece:
            head += f" | {piece}"
        blocks.append(f"{head}\n{node_text(card)}")
    return "\n\n".join(blocks)[:30000]


rows = []
card_rows = []
for item in QUESTIONS:
    gold = [] if not item["answerable"] else item["gold"]
    scores = item.get("scores") or {}
    matched = item.get("matched") or {}
    facts = item["facts"]
    names = uniq(labels.get(card.user_id or "", card.user_id or "") for card in gold)
    sources = uniq((card.source_input or "").strip() or "text" for card in gold)
    score_list = [scores.get(card.zettel_id, 0) for card in gold]
    texts = relevant_texts(gold, scores, matched)
    print(item["question_id"], "answerable=" + str(item["answerable"]).lower(), "узлов", len(gold))
    if texts:
        print(texts)

    rows.append({
        "question_id": item["question_id"],
        "question": item["question"],
        "multiproject_flag": item["flag"],
        "answerable": "true" if item["answerable"] else "false",
        "relevant_answer": item["answer"],
        "key_facts": as_json(facts),
        "relevant_luhmann": as_json(card.luhmann_id for card in gold if card.luhmann_id),
        "relevant_ids": as_json(card.zettel_id for card in gold),
        "rerank": as_json(score_list),
        "relevant_texts": texts,
        "relevant_project_name": as_json(names),
        "relevant_sources": as_json(sources),
        "reviewed": "false",
    })
    for card in gold:
        card_rows.append({
            "question_id": item["question_id"],
            "rerank": scores.get(card.zettel_id, ""),
            "matched": matched.get(card.zettel_id, ""),
            "luhmann_id": card.luhmann_id or "",
            "zettel_id": card.zettel_id,
            "project_name": labels.get(card.user_id or "", ""),
            "source_input": (card.source_input or "").strip() or "text",
            "text": node_text(card),
        })

dataset = pd.DataFrame(rows)
cards = pd.DataFrame(card_rows)

for row, item in zip(rows, QUESTIONS):
    assert row["relevant_answer"] == item["answer"]
    assert json.loads(row["key_facts"]) == item["facts"]
    ids = json.loads(row["relevant_ids"])
    found_scores = json.loads(row["rerank"])
    assert len(ids) == len(found_scores)
    assert all(score >= RERANK_MIN for score in found_scores)
    if not item["answerable"]:
        assert ids == []
        assert row["relevant_texts"] == ""
        assert json.loads(row["relevant_sources"]) == []
    print(row["question_id"], "ids=" + str(len(ids)), "rerank=" + row["rerank"])

display(dataset[["question_id", "relevant_answer", "rerank", "relevant_luhmann", "relevant_texts"]])
display(cards)


## 9. Запись

Ответ человека сохраняется как есть. `reviewed` остаётся `false`, пока узлы не проверены.


In [ ]:
OUTPUT = ROOT / "eval" / "synthetic_datasets" / "graphrag" / "qa_dataset_draft.xlsx"
DATASET_COLUMNS = [
    "question_id", "question", "multiproject_flag", "answerable",
    "relevant_answer", "key_facts", "relevant_luhmann", "relevant_ids",
    "rerank", "relevant_texts", "relevant_project_name", "relevant_sources", "reviewed",
]
CARD_COLUMNS = [
    "question_id", "rerank", "matched", "luhmann_id", "zettel_id",
    "project_name", "source_input", "text",
]


def load_output(path):
    if not path.exists():
        return pd.DataFrame(columns=DATASET_COLUMNS), pd.DataFrame(columns=CARD_COLUMNS)
    previous = pd.read_excel(path, sheet_name="dataset")
    try:
        previous_cards = pd.read_excel(path, sheet_name="cards")
    except ValueError:
        previous_cards = pd.DataFrame(columns=CARD_COLUMNS)
    return previous, previous_cards


def write_output(path, dataset_frame, cards_frame):
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        dataset_frame.reindex(columns=DATASET_COLUMNS).to_excel(writer, sheet_name="dataset", index=False)
        cards_frame.reindex(columns=CARD_COLUMNS).to_excel(writer, sheet_name="cards", index=False)


previous, previous_cards = load_output(OUTPUT)
ready_ids = set(dataset["question_id"].astype(str))
if not previous.empty and "question_id" in previous.columns:
    previous = previous[~previous["question_id"].astype(str).isin(ready_ids)]
if not previous_cards.empty and "question_id" in previous_cards.columns:
    previous_cards = previous_cards[~previous_cards["question_id"].astype(str).isin(ready_ids)]

dataset_out = pd.concat([previous, dataset], ignore_index=True)
cards_out = pd.concat([previous_cards, cards], ignore_index=True)
write_output(OUTPUT, dataset_out, cards_out)

written, written_cards = load_output(OUTPUT)
saved = written[written["question_id"].astype(str).isin(ready_ids)]
assert saved["relevant_answer"].astype(str).tolist() == [item["answer"] for item in QUESTIONS]
print(f"записано строк: {len(written)}, карточек: {len(written_cards)}")
print(OUTPUT)
display(saved[["question_id", "relevant_answer", "rerank", "relevant_luhmann", "reviewed"]])
